In [14]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import yaml
import time
import os
import hashlib
import sys
from pathlib import Path

try:
    script_dir = Path(__file__).resolve().parent
except NameError:
    script_dir = Path.cwd()

grandparent_dir = script_dir.parent.parent
sys.path.append(str(grandparent_dir))


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression, histogram_1D_config_creator
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [2]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/hal/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)

data_files = glob.glob(
  f"/workspace/root_files/hal/DATA_SEMILEPTONIC_NoSmearing/*.root"
  )
data_files += glob.glob(
  f"/workspace/root_files/kitt/DATA_SEMILEPTONIC_NoSmearing/*.root"
  )

# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

# Scaling factor
lumi_data = 1634433.9750
lumi_mc = 1444990.0313
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

mPi0 = 134.9768 # MeV
mPiCh = 139.57039 # MeV
mK0 = 497.611 # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonChTimeCMSignalFit - KaonNeTimeCMSignalFit")
    .Define("Delta_t_MC", "KaonChTimeCMMC - KaonNeTimeCMMC")
    .Define("Delta_t_rec", "KaonChTimeCMBoostLor - KaonNeTimeCMBoostLor")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[0]")
    .Define("trk2_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[1]")
    .Define("trk1_px_resolution", "trk1_matched[0] - trk1MC[0]")
    .Define("trk2_px_resolution", "trk2_matched[0] - trk2MC[0]")
    .Define("trk1_py_resolution", "trk1_matched[1] - trk1MC[1]")
    .Define("trk2_py_resolution", "trk2_matched[1] - trk2MC[1]")
    .Define("trk1_pz_resolution", "trk1_matched[2] - trk1MC[2]")
    .Define("trk2_pz_resolution", "trk2_matched[2] - trk2MC[2]")
    .Define("Kchrec_px_resolution", "Kchrec[0] - Kchmc[0]")
    .Define("Kchrec_py_resolution", "Kchrec[1] - Kchmc[1]")
    .Define("Kchrec_pz_resolution", "Kchrec[2] - Kchmc[2]")
    .Define("Kchrec_E_resolution", "Kchrec[3] - Kchmc[3]")
    .Define("Kchrec_x_resolution", "Kchrec[6] - Kchmc[6]")
    .Define("Kchrec_y_resolution", "Kchrec[7] - Kchmc[7]")
    .Define("Kchrec_z_resolution", "Kchrec[8] - Kchmc[8]")
    .Define("Kchboost_px_resolution", "Kchboost[0] - Kchmc[0]")
    .Define("Kchboost_py_resolution", "Kchboost[1] - Kchmc[1]")
    .Define("Kchboost_pz_resolution", "Kchboost[2] - Kchmc[2]")
    .Define("Kchboost_E_resolution", "Kchboost[3] - Kchmc[3]")
    .Define("Knerec_px_resolution", "Knerec[0] - Knemc[0]")
    .Define("Knerec_py_resolution", "Knerec[1] - Knemc[1]")
    .Define("Knerec_pz_resolution", "Knerec[2] - Knemc[2]")
    .Define("Knerec_E_resolution", "Knerec[3] - Knemc[3]")
    .Define("Knerec_x_resolution", "Knerec[6] - Knemc[6]")
    .Define("Knerec_y_resolution", "Knerec[7] - Knemc[7]")
    .Define("Knerec_z_resolution", "Knerec[8] - Knemc[8]")    
    .Define("KnetriKinFit_px_resolution", "KnetriKinFit[0] - Knemc[0]")
    .Define("KnetriKinFit_py_resolution", "KnetriKinFit[1] - Knemc[1]")
    .Define("KnetriKinFit_pz_resolution", "KnetriKinFit[2] - Knemc[2]")
    .Define("KnetriKinFit_E_resolution", "KnetriKinFit[3] - Knemc[3]")
    .Define("KnetriKinFit_x_resolution", "KnetriKinFit[6] - Knemc[6]")
    .Define("KnetriKinFit_y_resolution", "KnetriKinFit[7] - Knemc[7]")
    .Define("KnetriKinFit_z_resolution", "KnetriKinFit[8] - Knemc[8]")
    .Define("KchrecFit_px_resolution", "KchrecFit[0] - Kchmc[0]")
    .Define("KchrecFit_py_resolution", "KchrecFit[1] - Kchmc[1]")
    .Define("KchrecFit_pz_resolution", "KchrecFit[2] - Kchmc[2]")
    .Define("KchrecFit_E_resolution", "KchrecFit[3] - Kchmc[3]")
    .Define("KchrecFit_x_resolution", "KchrecFit[6] - Kchmc[6]")
    .Define("KchrecFit_y_resolution", "KchrecFit[7] - Kchmc[7]")
    .Define("KchrecFit_z_resolution", "KchrecFit[8] - Kchmc[8]")
    .Define("KchboostFit_px_resolution", "KchboostFit[0] - Kchmc[0]")
    .Define("KchboostFit_py_resolution", "KchboostFit[1] - Kchmc[1]")
    .Define("KchboostFit_pz_resolution", "KchboostFit[2] - Kchmc[2]")
    .Define("KchboostFit_E_resolution", "KchboostFit[3] - Kchmc[3]")
    .Define("KchboostFit_x_resolution", "KchboostFit[6] - Kchmc[6]")
    .Define("KchboostFit_y_resolution", "KchboostFit[7] - Kchmc[7]")
    .Define("KchboostFit_z_resolution", "KchboostFit[8] - Kchmc[8]")       
    .Define("KnerecFit_px_resolution", "KnerecFit[0] - Knemc[0]")
    .Define("KnerecFit_py_resolution", "KnerecFit[1] - Knemc[1]")
    .Define("KnerecFit_pz_resolution", "KnerecFit[2] - Knemc[2]")
    .Define("KnerecFit_E_resolution", "KnerecFit[3] - Knemc[3]")
    .Define("KnerecFit_x_resolution", "KnerecFit[6] - Knemc[6]")
    .Define("KnerecFit_y_resolution", "KnerecFit[7] - Knemc[7]")
    .Define("KnerecFit_z_resolution", "KnerecFit[8] - Knemc[8]")
    .Define("ip_bhabha_x_resolution", "Bx - ipmc[0]")
    .Define("ip_bhabha_y_resolution", "By - ipmc[1]")
    .Define("ip_bhabha_z_resolution", "Bz - ipmc[2]")
    .Define("ip_x_resolution", "ip[0] - ipmc[0]")
    .Define("ip_y_resolution", "ip[1] - ipmc[1]")
    .Define("ip_z_resolution", "ip[2] - ipmc[2]")
    .Define("ipFit_x_resolution", "ipFit[0] - ipmc[0]")
    .Define("ipFit_y_resolution", "ipFit[1] - ipmc[1]")
    .Define("ipFit_z_resolution", "ipFit[2] - ipmc[2]")
    .Define("KchrecPMFit_px_resolution", "KchrecPMFit[0] - Kchmc[0]")
    .Define("KchrecPMFit_py_resolution", "KchrecPMFit[1] - Kchmc[1]")
    .Define("KchrecPMFit_pz_resolution", "KchrecPMFit[2] - Kchmc[2]")
    .Define("KchrecPMFit_E_resolution", "KchrecPMFit[3] - Kchmc[3]")
    .Define("KchrecPMFit_x_resolution", "KchrecPMFit[6] - Kchmc[6]")
    .Define("KchrecPMFit_y_resolution", "KchrecPMFit[7] - Kchmc[7]")
    .Define("KchrecPMFit_z_resolution", "KchrecPMFit[8] - Kchmc[8]")
    .Define("KchboostPMFit_px_resolution", "KchboostPMFit[0] - Kchmc[0]")
    .Define("KchboostPMFit_py_resolution", "KchboostPMFit[1] - Kchmc[1]")
    .Define("KchboostPMFit_pz_resolution", "KchboostPMFit[2] - Kchmc[2]")
    .Define("KchboostPMFit_E_resolution", "KchboostPMFit[3] - Kchmc[3]")
    .Define("KchboostPMFit_x_resolution", "KchboostPMFit[6] - Kchmc[6]")
    .Define("KchboostPMFit_y_resolution", "KchboostPMFit[7] - Kchmc[7]")
    .Define("KchboostPMFit_z_resolution", "KchboostPMFit[8] - Kchmc[8]")
    .Define("KnerecNeutralFit_px_resolution", "KnerecNeutralFit[0] - Knemc[0]")
    .Define("KnerecNeutralFit_py_resolution", "KnerecNeutralFit[1] - Knemc[1]")
    .Define("KnerecNeutralFit_pz_resolution", "KnerecNeutralFit[2] - Knemc[2]")
    .Define("KnerecNeutralFit_E_resolution", "KnerecNeutralFit[3] - Knemc[3]")
    .Define("KnerecNeutralFit_x_resolution", "KnerecNeutralFit[6] - Knemc[6]")
    .Define("KnerecNeutralFit_y_resolution", "KnerecNeutralFit[7] - Knemc[7]")
    .Define("KnerecNeutralFit_z_resolution", "KnerecNeutralFit[8] - Knemc[8]")
    ###############################################################################################
    .Filter("mcflag == 0 && mctruth != -1")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonChTimeCMSignalFit - KaonNeTimeCMSignalFit")
    .Define("Delta_t_MC", "KaonChTimeCMMC - KaonNeTimeCMMC")
    .Define("Delta_t_rec", "KaonChTimeCMBoostLor - KaonNeTimeCMBoostLor")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[0]")
    .Define("trk2_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[1]")
    .Define("trk1_px_resolution", "trk1_matched[0] - trk1MC[0]")
    .Define("trk2_px_resolution", "trk2_matched[0] - trk2MC[0]")
    .Define("trk1_py_resolution", "trk1_matched[1] - trk1MC[1]")
    .Define("trk2_py_resolution", "trk2_matched[1] - trk2MC[1]")
    .Define("trk1_pz_resolution", "trk1_matched[2] - trk1MC[2]")
    .Define("trk2_pz_resolution", "trk2_matched[2] - trk2MC[2]")
    .Define("Kchrec_px_resolution", "Kchrec[0] - Kchmc[0]")
    .Define("Kchrec_py_resolution", "Kchrec[1] - Kchmc[1]")
    .Define("Kchrec_pz_resolution", "Kchrec[2] - Kchmc[2]")
    .Define("Kchrec_E_resolution", "Kchrec[3] - Kchmc[3]")
    .Define("Kchrec_x_resolution", "Kchrec[6] - Kchmc[6]")
    .Define("Kchrec_y_resolution", "Kchrec[7] - Kchmc[7]")
    .Define("Kchrec_z_resolution", "Kchrec[8] - Kchmc[8]")
    .Define("Kchboost_px_resolution", "Kchboost[0] - Kchmc[0]")
    .Define("Kchboost_py_resolution", "Kchboost[1] - Kchmc[1]")
    .Define("Kchboost_pz_resolution", "Kchboost[2] - Kchmc[2]")
    .Define("Kchboost_E_resolution", "Kchboost[3] - Kchmc[3]")
    .Define("Knerec_px_resolution", "Knerec[0] - Knemc[0]")
    .Define("Knerec_py_resolution", "Knerec[1] - Knemc[1]")
    .Define("Knerec_pz_resolution", "Knerec[2] - Knemc[2]")
    .Define("Knerec_E_resolution", "Knerec[3] - Knemc[3]")
    .Define("Knerec_x_resolution", "Knerec[6] - Knemc[6]")
    .Define("Knerec_y_resolution", "Knerec[7] - Knemc[7]")
    .Define("Knerec_z_resolution", "Knerec[8] - Knemc[8]")    
    .Define("KnetriKinFit_px_resolution", "KnetriKinFit[0] - Knemc[0]")
    .Define("KnetriKinFit_py_resolution", "KnetriKinFit[1] - Knemc[1]")
    .Define("KnetriKinFit_pz_resolution", "KnetriKinFit[2] - Knemc[2]")
    .Define("KnetriKinFit_E_resolution", "KnetriKinFit[3] - Knemc[3]")
    .Define("KnetriKinFit_x_resolution", "KnetriKinFit[6] - Knemc[6]")
    .Define("KnetriKinFit_y_resolution", "KnetriKinFit[7] - Knemc[7]")
    .Define("KnetriKinFit_z_resolution", "KnetriKinFit[8] - Knemc[8]")
    .Define("KchrecFit_px_resolution", "KchrecFit[0] - Kchmc[0]")
    .Define("KchrecFit_py_resolution", "KchrecFit[1] - Kchmc[1]")
    .Define("KchrecFit_pz_resolution", "KchrecFit[2] - Kchmc[2]")
    .Define("KchrecFit_E_resolution", "KchrecFit[3] - Kchmc[3]")
    .Define("KchrecFit_x_resolution", "KchrecFit[6] - Kchmc[6]")
    .Define("KchrecFit_y_resolution", "KchrecFit[7] - Kchmc[7]")
    .Define("KchrecFit_z_resolution", "KchrecFit[8] - Kchmc[8]")
    .Define("KchboostFit_px_resolution", "KchboostFit[0] - Kchmc[0]")
    .Define("KchboostFit_py_resolution", "KchboostFit[1] - Kchmc[1]")
    .Define("KchboostFit_pz_resolution", "KchboostFit[2] - Kchmc[2]")
    .Define("KchboostFit_E_resolution", "KchboostFit[3] - Kchmc[3]")
    .Define("KchboostFit_x_resolution", "KchboostFit[6] - Kchmc[6]")
    .Define("KchboostFit_y_resolution", "KchboostFit[7] - Kchmc[7]")
    .Define("KchboostFit_z_resolution", "KchboostFit[8] - Kchmc[8]")       
    .Define("KnerecFit_px_resolution", "KnerecFit[0] - Knemc[0]")
    .Define("KnerecFit_py_resolution", "KnerecFit[1] - Knemc[1]")
    .Define("KnerecFit_pz_resolution", "KnerecFit[2] - Knemc[2]")
    .Define("KnerecFit_E_resolution", "KnerecFit[3] - Knemc[3]")
    .Define("KnerecFit_x_resolution", "KnerecFit[6] - Knemc[6]")
    .Define("KnerecFit_y_resolution", "KnerecFit[7] - Knemc[7]")
    .Define("KnerecFit_z_resolution", "KnerecFit[8] - Knemc[8]")
    .Define("ip_bhabha_x_resolution", "Bx - ipmc[0]")
    .Define("ip_bhabha_y_resolution", "By - ipmc[1]")
    .Define("ip_bhabha_z_resolution", "Bz - ipmc[2]")
    .Define("ip_x_resolution", "ip[0] - ipmc[0]")
    .Define("ip_y_resolution", "ip[1] - ipmc[1]")
    .Define("ip_z_resolution", "ip[2] - ipmc[2]")
    .Define("ipFit_x_resolution", "ipFit[0] - ipmc[0]")
    .Define("ipFit_y_resolution", "ipFit[1] - ipmc[1]")
    .Define("ipFit_z_resolution", "ipFit[2] - ipmc[2]")
    .Define("KchrecPMFit_px_resolution", "KchrecPMFit[0] - Kchmc[0]")
    .Define("KchrecPMFit_py_resolution", "KchrecPMFit[1] - Kchmc[1]")
    .Define("KchrecPMFit_pz_resolution", "KchrecPMFit[2] - Kchmc[2]")
    .Define("KchrecPMFit_E_resolution", "KchrecPMFit[3] - Kchmc[3]")
    .Define("KchrecPMFit_x_resolution", "KchrecPMFit[6] - Kchmc[6]")
    .Define("KchrecPMFit_y_resolution", "KchrecPMFit[7] - Kchmc[7]")
    .Define("KchrecPMFit_z_resolution", "KchrecPMFit[8] - Kchmc[8]")
    .Define("KchboostPMFit_px_resolution", "KchboostPMFit[0] - Kchmc[0]")
    .Define("KchboostPMFit_py_resolution", "KchboostPMFit[1] - Kchmc[1]")
    .Define("KchboostPMFit_pz_resolution", "KchboostPMFit[2] - Kchmc[2]")
    .Define("KchboostPMFit_E_resolution", "KchboostPMFit[3] - Kchmc[3]")
    .Define("KchboostPMFit_x_resolution", "KchboostPMFit[6] - Kchmc[6]")
    .Define("KchboostPMFit_y_resolution", "KchboostPMFit[7] - Kchmc[7]")
    .Define("KchboostPMFit_z_resolution", "KchboostPMFit[8] - Kchmc[8]")
    .Define("KnerecNeutralFit_px_resolution", "KnerecNeutralFit[0] - Knemc[0]")
    .Define("KnerecNeutralFit_py_resolution", "KnerecNeutralFit[1] - Knemc[1]")
    .Define("KnerecNeutralFit_pz_resolution", "KnerecNeutralFit[2] - Knemc[2]")
    .Define("KnerecNeutralFit_E_resolution", "KnerecNeutralFit[3] - Knemc[3]")
    .Define("KnerecNeutralFit_x_resolution", "KnerecNeutralFit[6] - Knemc[6]")
    .Define("KnerecNeutralFit_y_resolution", "KnerecNeutralFit[7] - Knemc[7]")
    .Define("KnerecNeutralFit_z_resolution", "KnerecNeutralFit[8] - Knemc[8]")
    ###############################################################################################   
    .Filter("mcflag == 1 && mctruth != -1")
)

# Filtrowanie dla próbki semileptonicznej (dziedziczy wszystkie nowe kolumny m_*_hyp)
df_Semileptonic = df.Filter(
    "mcflag == 1 && (mctruth == 5 || mctruth == 0 || mctruth == -1)"
)

# df Signal
df_Signal = df.Filter(
    f"mcflag == 1 && mctruth == 1"
)

df_Background = df.Filter(
    f"mcflag == 1 && mctruth != 1 && mctruth != -1"
)

Przetestowano 351 plików. Poprawnych: 351, Odrzuconych: 0
Przetestowano 420 plików. Poprawnych: 420, Odrzuconych: 0
Lumi factor: 1.1311


In [15]:
base_path = "/workspace/root_notebooks/results"
cache_path = "/workspace/analysis_snapshot"
analysis_dir = "signal_separated_kin_fits"

current_date = time.strftime("%Y-%m-%d")

metadata = {
    "saved_errors": "Semileptonic",
    "cutset": "NONE",
    "luminosity_pb_MC": lumi_mc,
    "luminosity_pb_data": lumi_data,
    "scale_factor": scaling_factor,
}

analysis_hash = hashlib.md5(json.dumps(metadata, sort_keys=True).encode()).hexdigest()
metadata["analysis_hash"] = analysis_hash

current_result_dir = f"{base_path}/{analysis_dir}/{analysis_hash}"
cache_path = f"{cache_path}/{analysis_dir}/{analysis_hash}"
config_path = f"{base_path}/{analysis_dir}/config"

os.makedirs(current_result_dir, exist_ok=True)
with open(f"{current_result_dir}/analysis_config.json", "w") as f:
  json.dump(metadata, f, indent=4)

########################## LOAD HISTOGRAM CONFIGURATION ##############################
hist_config_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_1D.yaml"
with open(hist_config_file_path, "r") as f:
    hist_config = yaml.safe_load(f)
######################################################################################

############ LOAD VARIABLES RESOLUTIONS PER CUT HASH #################################
resolutions = {}
list_of_dirs = [
    f for f in os.listdir(config_path) if os.path.isdir(os.path.join(config_path, f))
]

for d in list_of_dirs:
  resolution_file_path = f"{config_path}/{d}/resolutions_config.yaml"
  with open(resolution_file_path, "r") as f:
      resolutions_config = yaml.safe_load(f)
  resolutions[d] = resolutions_config
######################################################################################

In [16]:
try:
  variables_1D = list(hist_config.get("histograms_1d", {}).keys())
  TRUTH_ONLY_VARS_1D = hist_config.get("truth_only_vars_1d", [])
  TRUTH_MCTRUTH_ID_1D = hist_config.get("truth_mctruth_id_1d", 1)

  variables_1D_titles = []
  unit_1D = []
  range_X_1D = {}
  modes = {}

  for var in variables_1D:
      variables_1D_titles.append(hist_config["histograms_1d"][var].get("title", ""))
      unit_1D.append(hist_config["histograms_1d"][var].get("unit", ""))
      modes[var] = list(hist_config["histograms_1d"][var].get("range_x", {}).keys())

      range_X_1D[var] = {}
      for mode in modes[var]:
          range_X_1D[var][mode] = hist_config["histograms_1d"][var].get("range_x", {}).get(mode, [])
      
except Exception as e:
  print(f"Error occurred: {e}")

print(resolutions)

{'62d706c5': {'variables': {'Knerec_5': {'mean': [-12.679050455692522, 0.12830371185775125], 'sigma_core': [34.78163288913888, 1.2943948665247262], 'sigma_tail': [57.27873648487422, 3.515443426018866], 'amplitude_ratio': [0.5372370372914536, 0.08388670073152998], 'fit_status': 0, 'chi2_ndf': 1.191494318870436}, 'Kchrec_5': {'mean': [-0.006743797627833002, 0.0025078805059232517], 'sigma_core': [0.505845271506128, 0.013675177468904842], 'sigma_tail': [1.0345059449651648, 0.0060881319472883], 'amplitude_ratio': [0.1769667488020766, 0.011605290263584492], 'fit_status': 0, 'chi2_ndf': 27.908540433814252}, 'Chi2SignalKinFit': {'mean': [10.62939273414455, 0.024591058102243935], 'sigma_core': [3.898956101045944, 0.014901898765034227], 'sigma_tail': [37.417126883232, 0.3167003781062796], 'amplitude_ratio': [0.7155957670578151, 0.0022703096139933265], 'fit_status': 0, 'chi2_ndf': 314.4296530082437}, 'Chi2NeutralKinFit': {'mean': [7.420089857186363, 0.024374704036730144], 'sigma_core': [4.0423891

In [17]:
## Prepare 1D histogram model on levels:
## cut_hash --> mode --> variable

def prepare_hist_model(cut_hash: list, vars: list, vars_titles: list, unit: list, range_X: dict, resolutions: dict) -> dict:
  hist_model = {}

  if isinstance(cut_hash, str):
      cut_hash = [cut_hash]

  for cut in cut_hash:
      hist_model[cut] = {}

      for var in variables_1D:
        for mode in modes[var]:
          if mode not in hist_model[cut]:
              hist_model[cut][mode] = {}

          ## Prepare sigma for the variable
          sigma_core = resolutions.get(cut, {}).get('variables', {}).get(var, {}).get('sigma_core', [1.0])[0]
          sigma_tail = resolutions.get(cut, {}).get('variables', {}).get(var, {}).get('sigma_tail', [1.0])[0]

          sigma = min(sigma_core, sigma_tail)

          ## Calculate the range length for the variable in the given mode
          range_X_min, range_X_max = range_X[var].get(mode, [0, 1])
          range_len = range_X_max - range_X_min

          title = f"{vars_titles[vars.index(var)]} [{unit[vars.index(var)]}]" if var in vars else ""

          hist_model[cut][mode][var] = {
              "title": title,
              "range_x": range_X[var].get(mode, []),
              "n_bins": int(range_len / sigma) if sigma > 0 else range_len,
              "resolution": sigma
          }

  return hist_model

In [18]:
histo1Dmodel = prepare_hist_model('62d706c5', variables_1D, variables_1D_titles, unit_1D, range_X_1D, resolutions)

print(histo1Dmodel)

{'62d706c5': {'full': {'Knerec_5': {'title': 'm^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]', 'range_x': [-400, 200], 'n_bins': 17, 'resolution': 34.78163288913888}, 'KnerecSix_5': {'title': 'm^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]', 'range_x': [-400, 200], 'n_bins': 600, 'resolution': 1.0}, 'Kchrec_5': {'title': 'm^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]', 'range_x': [-200, 200], 'n_bins': 790, 'resolution': 0.505845271506128}, 'Chi2SignalKinFit': {'title': '#chi^{2}_{+-00} []', 'range_x': [0, 500], 'n_bins': 128, 'resolution': 3.898956101045944}, 'Chi2NeutralKinFit': {'title': '#chi^{2}_{00} []', 'range_x': [0, 500], 'n_bins': 123, 'resolution': 4.042389132417685}, 'Chi2PMKinFit': {'title': '#chi^{2}_{+-} []', 'range_x': [0, 100], 'n_bins': 76, 'resolution': 1.3024348097844094}, 'MassPi01Fit': {'title': 'm^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]', 'range_x': [-400, 400], 'n_bins': 482, 'resolution': 1.656511990461097}, 'MassPi02Fit': {'title': 'm^{inv,fit,+-00}_{#gamma#gam

### Get 2D histograms for each cut

In [ ]:
variables_2D = [["Knerec_5", "KnerecSix_5"], 
                ["Chi2SignalKinFit", "Chi2NeutralKinFit"], 
                ["Chi2SignalKinFit", "Chi2PMKinFit"],
                ["Chi2PMKinFit", "Chi2NeutralKinFit"],
                ["MassPi01Fit", "MassPi02Fit"],
                ["MassPi01", "MassPi02"],
                ["MassPi0NeutralFit1", "MassPi0NeutralFit2"],
                ["MassPi0PMFit1", "MassPi0PMFit2"],
                ["Delta_t_MC", "Delta_t_fit_resolution"],
                ["Delta_t_MC", "Delta_t_rec_resolution"]]
variables_2D_titles = [
                       ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{00} [-]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{+-} [-]"],
                       ["#chi^{2}_{+-} [-]", "#chi^{2}_{00} [-]"],
                       ["m^{inv,+-00}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,+-00}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv,00}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,00}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv,+-}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,+-}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["#Delta t^{MC} [#tau_{S}]", "#Delta t^{fit} - #Delta t^{MC} [#tau_{S}]"],
                       ["#Delta t^{MC} [#tau_{S}]", "#Delta t^{rec} - #Delta t^{MC} [#tau_{S}]"]
                       ]
range_X_2D = [[(100, -400, 200), (100, -400, 200)], 
              [(100, 0, 100), (100, 0, 100)], 
              [(100, 0, 100), (100, 0, 30)],
              [(100, 0, 30), (100, 0, 100)], 
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -300, 300), (100, -20, 20)],
              [(100, -300, 300), (100, -20, 20)]]

# Zmienne 2D widoczne TYLKO dla jednego, wybranego mctruth - analogicznie do 1D.
# Przynależność do grupy "truth-only" sprawdzana jest po kluczu "x_y" (var_key).
TRUTH_ONLY_VARS_2D = set([("Delta_t_MC", "Delta_t_fit_resolution"), ("Delta_t_MC", "Delta_t_rec_resolution")])
TRUTH_MCTRUTH_ID_2D = 1

In [ ]:
## 2D histograms
hist_2D = {}
modes = {}

# Definicja zakresów dla trybów: (nbinsX, xlow, xhigh, nbinsY, ylow, yhigh)
modes["full"] = [(100, -400, 200, 100, -400, 200), 
                 (100, 0, 100, 100, 0, 100), 
                 (100, 0, 100, 100, 0, 30),
                 (100, 0, 30, 100, 0, 100),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -300, 300, 100, -20, 20),
                 (100, -300, 300, 100, -20, 20)
                 ]
modes["zoom"] = [(100, -100, 100, 100, -100, 100), 
                 (100, 0, 50, 100, 0, 50), 
                 (100, 0, 50, 100, 0, 15),
                  (100, 0, 15, 100, 0, 50),
                 (100, -50, 50, 100, -50, 50),
                 (100, -50, 50, 100, -50, 50),
                (100, -50, 50, 100, -50, 50),
                (100, -50, 50, 100, -50, 50),
                (100, -100, 100, 100, -10, 10),
                (100, -100, 100, 100, -10, 10)]

(vars_2D_main, titles_2D_main, modes_2D_main), (vars_2D_truth, titles_2D_truth, modes_2D_truth) = \
    split_variable_group(
        variables_2D, variables_2D_titles, modes,
        is_truth_only=lambda pair: "_".join(pair) in TRUTH_ONLY_VARS_2D
    )

hist_2D = fill_2d_histograms(
    df, df_data, channName, vars_2D_main, titles_2D_main, modes_2D_main,
    total_cut, scaling_factor, compute_mc_sum=True
)

hist_2D_truth = {}
if vars_2D_truth:
  truth_channels_2D = {TRUTH_MCTRUTH_ID_2D: channName[TRUTH_MCTRUTH_ID_2D]}
  hist_2D_truth = fill_2d_histograms(
      df, df_data, truth_channels_2D, vars_2D_truth, titles_2D_truth, modes_2D_truth,
      total_cut, compute_mc_sum=False
  )

In [ ]:
root_file_name = "separated_kin_fits_2D_variables.root"
hist_2D, canva = draw_and_save_2d_histograms(
    hist_2D, vars_2D_main, modes_2D_main, current_result_dir, cut_hash,
    cuts_json_str, cut_list, root_file_name
)

if vars_2D_truth:
  root_file_name_truth = "separated_kin_fits_2D_truth_variables.root"
  hist_2D_truth, canva_truth = draw_and_save_2d_histograms(
      hist_2D_truth, vars_2D_truth, modes_2D_truth, current_result_dir, cut_hash,
      cuts_json_str, cut_list, root_file_name_truth
  )

for canva in canvas:
  canva.Close()
if vars_1D_truth:
  for canva in canvas_truth:
    canva.Close()

In [ ]:
import pandas as pd
import json
import hashlib

df = pd.read_csv("results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv")

hash_dict = df[["hash", "cuts_description"]]
hash_dict = hash_dict.set_index("hash").to_dict()["cuts_description"]

# Zdefiniuj swoje pełne listy cięć i parametry tak jak dotychczas:
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"

cut1 = 8.086
cut2 = 11.515

# Hash list order
full_cut_list = [
    "abs(Kchrec_5) <= 3 * 0.8090",
    "ellipse_cut_func(Knerec_5, KnerecSix_5)",
    f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
    f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
    f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
    "Chi2SignalKinFit < 30.0",
]

hash_order = []
for i in range(-1, len(full_cut_list)):

  if i == -1:
    current_cut_list = []  # Stan bazowy bez cięć
    step_name = "00_no_cuts"
  else:
    current_cut_list = full_cut_list[: i + 1]
    step_name = f"step_{i+1:02d}"

  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = (
        "nocuts"
        if not current_cut_list
        else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
    )

  hash_order.append(cut_hash)

In [ ]:
import json
import ROOT

# Getting purity, efficiency and number of events for each cut
results_file = ROOT.TFile("results/signal_separated_kin_fits/2026-10-22/separated_kin_fits_1D_variables.root", "READ")

no_cuts_hash = "nocuts"
num_of_events = {}
purity = {}

for hash, cut in hash_dict.items():
    obj_integrals = results_file.Get(f"{hash}/ChannelIntegrals")
    obj_purity = results_file.Get(f"{hash}/Purity")
    if obj_integrals:
      if hasattr(obj_integrals, "GetTitle"):
        num_of_events[hash] = json.loads(obj_integrals.GetTitle())

    if obj_purity:
      if hasattr(obj_purity, "GetVal"):
        purity[hash] = obj_purity.GetVal()

In [ ]:
import numpy as np
import ROOT

x, y = [], []

# Pamiętaj o użyciu innej nazwy zmiennej niż 'hash' (wbudowana funkcja Pythona)
for i, curr_hash in enumerate(hash_order):
  x.append(float(i))
  y.append(float(purity[curr_hash]))

canva = ROOT.TCanvas("canva", "Purity vs Cut Step", 800, 600)
canva.SetLogy(True)  # Skala logarytmiczna
ROOT.gPad.SetGrid(True, True)

# Liczba binów musi odpowiadać liczbie elementów w y
n_bins = len(y)
h_purity = ROOT.TH1D(
    "h_purity", "Czystość w kolejnych krokach cięć;Krok selekcji;Czystość", n_bins, 0, n_bins
)

# Przykładowe etykiety dla osi X (dostosuj do liczby kroków w hash_order)
labels = ["No cuts", "Cut 1", "Cut 2", "Cut 3", "Cut 4", "Cut 5", "Cut 6"]

for i, val in enumerate(y):
  h_purity.SetBinContent(i + 1, val)
  # Dodanie czytelnych opisów do każdego binu (indeksowanie w ROOT od 1)
  if i < len(labels):
    h_purity.GetXaxis().SetBinLabel(i + 1, labels[i])

h_purity.SetStats(0)  # Usuwa stat box
h_purity.SetFillColor(ROOT.kAzure + 4)

h_purity.Draw("HIST")

# Obrócenie etykiet osi X, jeśli nazwy cięć są dłuższe
h_purity.GetXaxis().SetLabelSize(0.035)
canva.SetBottomMargin(0.2)  # Zostawienie miejsca na dole na opisy

canva.Draw()

### Test of cuts for separated kinematic fit

In [8]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = ["abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {cut1}, {cut2})",
            "Chi2NeutralKinFit < 30.0 && Chi2PMKinFit < 30.0"
            ]

In [9]:
# Wypuszczasz dane po cięciach bazowych do lokalnego pliku raz przed pętlą:
# 1. Bierzemy wszystkie cięcia OPRÓCZ ostatnich dwóch (do optymalizacji)
base_cuts = cut_list[:-2]
print("Cięcia bazowe do cache:", base_cuts)

cuts_json_str = json.dumps(base_cuts, indent=2)
cut_hash = "nocuts" if not base_cuts else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]

total_base_cut = " && ".join([f"({cut})" for cut in base_cuts])

columns_to_save = [
    # Zdefiniowane zmienne korelacji mas
    "Kchrec_5",
    "Knerec_5",
    "KnerecSix_5",
    "KnetriKinFit_5",
    "MassPi01",
    "MassPi02",
    "MassPi01Fit",
    "MassPi02Fit",
    "MassPi0NeutralFit1",
    "MassPi0NeutralFit2",
    "MassPi0PMFit1",
    "MassPi0PMFit2",
    
    # Cylindryczne i współrzędne
    "rho_ch",
    "rho_ne",
    "z_ch",
    "z_ne",
    
    # Omega-pi0 geometrical cut variables
    "bhabha_IP",
    "bhabha_IP_changed",
    "rho_00_closest",
    "rho_pm_closest",
    "rho_omega",
    "z_00_closest",
    "z_pm_closest",
    
    # Bad tracks geometrical cut variables
    "rho_pm_00",
    "z_pm_00",
    "cos_trk1_K",
    "cos_trk2_K",
    
    # Delta t
    "Delta_t_fit",
    "Delta_t_MC",
    "Delta_t_rec",
    
    # Resolutions i matching
    "Delta_t_fit_resolution",
    "Delta_t_rec_resolution",
    "trk1_matched",
    "trk2_matched",
    "trk1_px_resolution",
    "trk2_px_resolution",
    "trk1_py_resolution",
    "trk2_py_resolution",
    "trk1_pz_resolution",
    "trk2_pz_resolution",

    # Mctruth
    "mctruth",

    # Chi2
    "Chi2SignalKinFit",
    "Chi2NeutralKinFit",
    "Chi2PMKinFit"
]

# Konwersja listy Pythona na wektor C++ wymagany przez RDataFrame.Snapshot w PyROOT
col_vec = ROOT.std.vector("string")()
for col in columns_to_save:
    col_vec.push_back(col)

# 3. Filtrujemy i robimy Snapshot
os.makedirs(f"{cache_path}/{cut_hash}", exist_ok=True)

cache_MC_name = f"{cache_path}/{cut_hash}/local_root_cache.root"
cache_data_name = f"{cache_path}/{cut_hash}/local_root_cache_data.root"

if not os.path.exists(cache_MC_name):
  df_base = df.Filter(total_base_cut)
  df_base.Snapshot(
      "h1", cache_MC_name, col_vec
  )

if not os.path.exists(cache_data_name):
  df_base = df_data.Filter(total_base_cut)
  df_base.Snapshot(
      "h1", cache_data_name, col_vec
  )

Cięcia bazowe do cache: ['abs(Kchrec_5) <= 3 * 0.8090', 'ellipse_cut_func(Knerec_5, KnerecSix_5)', '(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0 && rho_omega > 1.5) || !(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0)', '(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45 && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45)']


### Tests of separated cuts

In [10]:
# Loading cached RDataFrames from local root cache files
df_MC_cache = ROOT.RDataFrame("tree_name", cache_MC_name)
df_data_cache = ROOT.RDataFrame("tree_name", cache_data_name)

In [13]:
import ROOT
from utils.kinfit_scan import run_kinfit_scan

if not ROOT.IsImplicitMTEnabled():
    ROOT.EnableImplicitMT()

# cut1 and cut2 scan a fixed 0..200 range; cut3 and cut4 use data maxima.
scan_config = {
    "coarse_points_per_cut": 31,
    "fine_num_steps": 5,
    "fine_step_factor": 0.1,
    "final_metric_mode": "target_distance",
    "target_purity_start": 0.2218,
    "target_efficiency_start": 0.5593,
    "target_purity": 0.72,
    "target_efficiency": 0.4906,
    "min_efficiency": 0.0,
    "min_purity": 0.0,
    "output_file": f"{current_result_dir}/optimization_scan_results_changed.root",
}
optimization_results = run_kinfit_scan(df_MC_cache, ROOT, scan_config)
results_step1_coarse = optimization_results["results_step1_coarse"]
results_step1_fine = optimization_results["results_step1_fine"]
results_step2_coarse = optimization_results["results_step2_coarse"]
results_step2_fine = optimization_results["results_step2_fine"]
best_cut1 = optimization_results["best_cut1"]
best_cut2 = optimization_results["best_cut2"]
best_cut3 = optimization_results["best_cut3"]
best_cut4 = optimization_results["best_cut4"]
best_eff_1 = optimization_results["best_eff_1"]
best_pur_1 = optimization_results["best_pur_1"]
best_eff_2b = optimization_results["best_eff_2b"]
best_pur_2b = optimization_results["best_pur_2b"]


Globalne zakresy progów (od zera): cut1 <= 200, cut2 <= 200, cut3 <= 500, cut4 <= 100
Uruchamiam Etap1_Coarse: 961 punktów 2D...
Etap1_Coarse: cut1=46.6667, cut2=86.6667; eff=0.5460, pur=0.0139
Uruchamiam Etap1_Fine: 121 punktów 2D...
Etap1_Fine: cut1=44, cut2=90; eff=0.5427, pur=0.0142
Uruchamiam Etap2_Coarse: 961 punktów 2D...
Etap2_Coarse: cut3=16.6667, cut4=3.33333; eff=0.2202, pur=0.4251
Uruchamiam Etap2_Fine: 121 punktów 2D...
Etap2_Fine: cut3=13.3333, cut4=2.33333; eff=0.1837, pur=0.4851
Wyniki, mapy 2D i canvasy zapisano do: /workspace/root_notebooks/results/signal_separated_kin_fits/c97f42f6a601db1c64b593e259e04dec/optimization_scan_results_changed.root
Finalne cięcia: cut1=44, cut2=90, cut3=13.3333, cut4=2.33333
Efficiency=0.1837, Purity=0.4851


In [20]:
## Apply cuts
optimization_file_path = f"{current_result_dir}/optimization_scan_results_changed.root"
optimization_file = ROOT.TFile.Open(optimization_file_path)

tree_best_point = optimization_file.Get("scan_BestPoint")

tree_best_point.GetEntry(0)

cut3 = tree_best_point.GetBranch("cut3").GetLeaf("cut3").GetValue()
cut4 = tree_best_point.GetBranch("cut4").GetLeaf("cut4").GetValue()

## Set cut list full

cut1 = tree_best_point.GetBranch("cut1").GetLeaf("cut1").GetValue()
cut2 = tree_best_point.GetBranch("cut2").GetLeaf("cut2").GetValue()
cut3 = tree_best_point.GetBranch("cut3").GetLeaf("cut3").GetValue()
cut4 = tree_best_point.GetBranch("cut4").GetLeaf("cut4").GetValue()

full_cut_list = [f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {cut1}, {cut2})",
            f"Chi2PMKinFit < {cut4}",
            f"Chi2NeutralKinFit < {cut3}"]

print(full_cut_list)

num_bkg = [df_MC_cache.Filter(f"mctruth != 1").Count()]
num_signal = [df_MC_cache.Filter(f"mctruth == 1").Count()]
efficiency = []
purity = []
relative_efficiency = []

total_cut = ""
for cut in full_cut_list:
    if total_cut:
        total_cut += " && "
    total_cut += f"({cut})"

    num_signal.append(df_MC_cache.Filter(f"mctruth == 1 && {total_cut}").Count())
    num_bkg.append(df_MC_cache.Filter(f"mctruth != 1 && {total_cut}").Count())

total_num = num_signal + num_bkg

print(f"Total num: {total_num}")

ROOT.RDF.RunGraphs(total_num)

num_signal = [num.GetValue() for num in num_signal]
num_bkg = [num.GetValue() for num in num_bkg]

total_num = num_signal + num_bkg

for i in range(1, len(num_signal)):
    efficiency.append(num_signal[i] / num_signal[0])
    purity.append(num_signal[i] / (num_signal[i] + num_bkg[i]))

    relative_efficiency.append(num_signal[i] / num_signal[i-1])
    print(f"Cut {i}: Num Signal = {num_signal[i]}, Num Bkg = {num_bkg[i]}, Efficiency = {efficiency[-1]}, Purity = {purity[-1]}, Relative Efficiency = {relative_efficiency[-1]}")


['m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, 44.00000000000001, 90.0)', 'Chi2PMKinFit < 2.3333333333333335', 'Chi2NeutralKinFit < 13.333333333333334']
Total num: [<cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c78a7c580>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c837711e0>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c81e98e80>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c79f74210>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c86551080>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c7a02fe70>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c80781340>, <cppyy.gbl.ROOT.RDF.RResultPtr<ULong64_t> object at 0x555c7804c6b0>]
Cut 1: Num Signal = 153207, Num Bkg = 10611111, Efficiency = 0.5426599037991542, Purity = 0.01423285711180216, Relative Efficiency = 0.5426599037991542
Cut 2: Num Signal = 115844, Num Bkg = 7419703, Efficiency = 0.41031998469854, Purity = 0.015373004773243402, Relative Effic

In [51]:
import ROOT

optimization_file_path = f"{current_result_dir}/optimization_scan_results.root"
optimization_file = ROOT.TFile.Open(optimization_file_path)  # Otwórz w trybie zapisu!

canvas_names = ["canvas_Etap1_Coarse", "canvas_Etap1_Fine", "canvas_Etap2_Coarse", "canvas_Etap2_Fine"]

for c_name in canvas_names:
    canva = optimization_file.Get(c_name)
    index = 0
    if not canva:
        continue
                
  # Pobieramy listę obiektów bezpośrednio z płótna
    primitives = canva.GetListOfPrimitives()
    if primitives:
        for obj in primitives:
            # Sytuacja A: Płótno jest podzielone na mniejsze pady (TPad)
            if obj.InheritsFrom("TPad"):
                obj.SetLogz(0)
                sub_primitives = obj.GetListOfPrimitives()
                if sub_primitives:
                    for sub_obj in sub_primitives:
                        if sub_obj.InheritsFrom("TH1"):
                            titles = ["Efficiency", "Purity", "Significance", "Distance from optimum"]
                            sub_obj.SetTitle(titles[index])
                            index += 1
            
            # Sytuacja B: Histogram leży bezpośrednio na płótnie
            elif obj.InheritsFrom("TH1"):
                obj.SetTitle("; Nowa oś X; Nowa oś Y")
        
    # Odświeżamy płótno i zapisujemy zmiany z powrotem do pliku ROOT
    canva.Draw()

In [21]:
############# FIRST CUT LIST FOR 000 TAG ################
from utils.hist_utils import fill_1D_histogram_simple

df = df_MC_cache
df_data = df_data_cache

# --- GŁÓWNA PĘTLA BUDUJĄCA CIĘCIA KROK PO KROKU ---
# Możesz zacząć od indeksu -1 (czyli stan "beż żadnych cięć" / nocuts)
# lub od i = 0 (jeśli chcesz zacząć od razu od pierwszego cięcia).
for i in range(-1, len(full_cut_list)):

  # Wycinamy podzbiór cięć od początku do bieżącego kroku
  if i == -1:
    current_cut_list = []  # Stan bazowy bez cięć
    step_name = "00_no_cuts"
  else:
    current_cut_list = full_cut_list[: i + 1]
    step_name = f"step_{i+1:02d}"

  print(
      f"\n[INFO] Przetwarzanie kroku: {step_name} (liczba aktywnych cięć:"
      f" {len(current_cut_list)})"
  )

  # Przygotowanie metadanych, hashu i unikalnego katalogu dla tego kroku
  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = (
      "nocuts"
      if not current_cut_list
      else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
  )

  # Możesz opcjonalnie rozbić wyniki na podkatalogi dla każdego kroku, np.:
  # step_result_dir = os.path.join(current_result_dir, step_name)
  step_result_dir = (
      current_result_dir  # Lub zapisywać w jednym miejscu z hashem w nazwie pliku
  )

  total_cut = build_cut_expression(current_cut_list)

  # Rozdzielczości i model histogramu zależą od bieżącego zestawu cięć.
  histo1Dmodel = prepare_hist_model(
      cut_hash,
      variables_1D,
      variables_1D_titles,
      unit_1D,
      range_X_1D,
      resolutions,
  )
  model_for_cut = histo1Dmodel[cut_hash]

  truth_only_vars = set(TRUTH_ONLY_VARS_1D)
  main_indices = [
      index for index, variable in enumerate(variables_1D)
      if variable not in truth_only_vars
  ]
  truth_indices = [
      index for index, variable in enumerate(variables_1D)
      if variable in truth_only_vars
  ]
  vars_1D_main = [variables_1D[index] for index in main_indices]
  titles_1D_main = [variables_1D_titles[index] for index in main_indices]
  vars_1D_truth = [variables_1D[index] for index in truth_indices]
  titles_1D_truth = [variables_1D_titles[index] for index in truth_indices]

  def modes_for_variables(variables):
    return {
        mode: {}
        for mode, variables_by_mode in model_for_cut.items()
        if all(variable in variables_by_mode for variable in variables)
    }

  modes_1D_main = modes_for_variables(vars_1D_main)
  modes_1D_truth = modes_for_variables(vars_1D_truth)

  # Wypełnianie histogramów głównych: Data + kanały MC + suma MC
  hist_1D = fill_1D_histogram_simple(
      df,
      df_data,
      channName,
      histo1Dmodel,
      cut_hash,
      total_cut,
      variables=vars_1D_main,
      scaling_factor=scaling_factor,
      compute_mc_sum=True,
  )

  # Wypełnianie histogramów truth (jeśli istnieją)
  hist_1D_truth = {}
  if vars_1D_truth:
    truth_channels_1D = {
        TRUTH_MCTRUTH_ID_1D: channName[TRUTH_MCTRUTH_ID_1D]
    }
    hist_1D_truth = fill_1D_histogram_simple(
        df,
        df_data,
        truth_channels_1D,
        histo1Dmodel,
        cut_hash,
        total_cut,
        variables=vars_1D_truth,
        compute_mc_sum=False,
    )

  # Rysowanie i zapis do dedykowanych plików ROOT / PDF
  # (zakładam, że Twoja funkcja draw_and_save przyjmuje cut_hash i zapisuje pliki)
  root_file_name = f"separated_kin_fits_1D_variables.root"
  hist_1D, canvas, legends = draw_and_save_1d_histograms(
      hist_1D,
      vars_1D_main,
      modes_1D_main,
      channColor,
      channTitle,
      step_result_dir,
      cut_hash,
      cuts_json_str,
      current_cut_list,
      root_file_name,
  )

  # if vars_1D_truth:
  #   root_file_name_truth = (
  #       f"separated_kin_fits_1D_truth_variables.root"
  #   )
  #   hist_1D_truth, canvas_truth, legends_truth = draw_and_save_1d_histograms(
  #       hist_1D_truth,
  #       vars_1D_truth,
  #       modes_1D_truth,
  #       channColor,
  #       channTitle,
  #       step_result_dir,
  #       cut_hash,
  #       cuts_json_str,
  #       current_cut_list,
  #       root_file_name_truth,
  #   )

  # Sprzątanie obiektów Canvas z pamięci ROOT-a
  for canva in canvas:
    canva.Close()
  # if vars_1D_truth:
  #   for canva in canvas_truth:
  #     canva.Close()

print(
    "[INFO] Pętla sekwencyjnego nakładania cięć została pomyślnie zakończona!"
)


[INFO] Przetwarzanie kroku: 00_no_cuts (liczba aktywnych cięć: 0)
[INFO] Zaktualizowano wpis dla hashu nocuts w rejestrze w /workspace/root_notebooks/results/signal_separated_kin_fits/c97f42f6a601db1c64b593e259e04dec/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_01 (liczba aktywnych cięć: 1)
[INFO] Zaktualizowano wpis dla hashu 0e039090 w rejestrze w /workspace/root_notebooks/results/signal_separated_kin_fits/c97f42f6a601db1c64b593e259e04dec/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_02 (liczba aktywnych cięć: 2)
[INFO] Zaktualizowano wpis dla hashu 6765aec4 w rejestrze w /workspace/root_notebooks/results/signal_separated_kin_fits/c97f42f6a601db1c64b593e259e04dec/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_03 (liczba aktywnych cięć: 3)
[INFO] Zaktualizowano wpis dla hashu 7a9b5775 w rejestrze w /workspace/root_notebooks/results/signal_separated_kin_fits/c97f42f6a601db1c64b593e259e04dec/analysis_registry.csv
[INFO] Pętla sekwencyjnego nakładania cięć 